# Spectra and inertia

The eigenvalues of a symmetric matrix are real, and three numbers summarise
their signs: the **inertia** (n₊, n₋, n₀), the counts of positive, negative and
zero eigenvalues. Sylvester's law of inertia says that a congruence
A → XᵀAX, with X nonsingular, keeps those counts. That is why an LDLᵀ
factorization can read the inertia off the signs of D, without computing a
single eigenvalue.

Inertia decides which algorithms apply. Cholesky and CG need n₋ = n₀ = 0. An
indefinite matrix needs LDLᵀ or MINRES, and a constrained optimization problem
needs a specific inertia in its KKT matrix. A test matrix with a *chosen*
inertia is therefore the right input for testing all of these.

This notebook:

1. **Reads spectra** of catalog and built-to-order matrices, with their
   inertia.
2. **Builds a spectrum to order** with `randsym(n, eigenvalues, seed=...)`.
3. **Slices a spectrum** by counting the negative pivots of A − σI, which
   locates eigenvalues with no eigensolver.
4. **Computes inertia in low precision.** It asks when a format gets the sign
   of a small eigenvalue wrong, across the same eight formats as the
   conditioning notebook.
5. **Bounds a spectrum** with Gershgorin discs.

Every random matrix has a seed, so the figures are the same on every build.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import ListedColormap
from matplotlib.patches import Circle

import mtl5

g = mtl5.generators

# Chart styling, shared with the other notebooks.
SURFACE = "#fcfcfb"
INK, INK_2, RULE = "#0b0b0b", "#52514e", "#a9a8a3"
SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]

plt.rcParams.update(
    {
        "figure.facecolor": SURFACE,
        "axes.facecolor": SURFACE,
        "axes.edgecolor": "#d8d7d2",
        "axes.labelcolor": INK_2,
        "axes.titlecolor": INK,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.grid": True,
        "grid.color": "#e8e7e2",
        "grid.linewidth": 0.8,
        "text.color": INK,
        "xtick.color": INK_2,
        "ytick.color": INK_2,
        "font.size": 10,
        "figure.dpi": 120,
    }
)

# The sign of an eigenvalue gets a colour AND a marker, so it never rests on
# colour alone.
SIGN = {
    "negative": dict(color=SERIES[1], marker="v"),
    "zero": dict(color=INK_2, marker="o"),
    "positive": dict(color=SERIES[0], marker="^"),
}

FORMATS = ["fp16", "posit16", "takum16", "f32", "posit32", "takum32", "f64", "dd_cascade"]

print(mtl5.__version__)

## 1. Reading a spectrum

Five symmetric matrices, each with a spectrum worth knowing:

- **`randspd(12, eigenvalues)`:** SPD by construction, here with eigenvalues
  spread geometrically from 10⁻² to 1.
- **`randsym(12, eigenvalues)`:** a chosen *indefinite* spectrum, with inertia
  (7, 4, 1), one exact zero included.
- **`clement(12)`:** eigenvalues exactly ±1, ±3, …, ±11, so its inertia is
  (6, 6, 0) for even n.
- **`wilkinson(13)`:** pairs of nearly equal eigenvalues, and a single negative
  one.
- **`rosser()`:** the classic 8×8 test, with a double eigenvalue, a zero
  eigenvalue and a nearly equal pair.

Each row plots the eigenvalues scaled by the largest |λ|, so the five fit on
one axis, beside the inertia `mtl5.inertia` reports.

In [ ]:
def sign_of(lam, tol):
    return np.where(lam > tol, "positive", np.where(lam < -tol, "negative", "zero"))


matrices = {
    "randspd(12, geometric 1e-2..1)": g.randspd(12, list(np.geomspace(1e-2, 1, 12)), seed=1),
    "randsym(12, inertia (7, 4, 1))": g.randsym(
        12, [-4.0, -2.5, -1.0, -0.5, 0.0, 0.5, 1.0, 1.5, 2.0, 3.0, 4.0, 5.0], seed=2
    ),
    "clement(12)": g.clement(12),
    "wilkinson(13)": g.wilkinson(13),
    "rosser()": g.rosser(),
}

fig, (ax, axb) = plt.subplots(
    1, 2, figsize=(12, 3.8), gridspec_kw={"width_ratios": [3, 1.2]}, constrained_layout=True
)
for row, (name, A) in enumerate(matrices.items()):
    lam = mtl5.eigvalsh(A)
    scale = np.max(np.abs(lam))
    signs = sign_of(lam, 1e-12 * scale)
    for s, style in SIGN.items():
        pick = signs == s
        ax.scatter(
            lam[pick] / scale,
            np.full(pick.sum(), row),
            s=42,
            zorder=3,
            edgecolor=SURFACE,
            linewidth=1,
            **style,
        )
    inertia = mtl5.inertia(A)
    left = 0
    for s in ("negative", "zero", "positive"):
        count = inertia[s]
        axb.barh(
            row,
            count,
            left=left,
            height=0.55,
            color=SIGN[s]["color"],
            edgecolor=SURFACE,
            linewidth=2,
        )
        if count:
            axb.text(
                left + count / 2,
                row,
                str(count),
                ha="center",
                va="center",
                fontsize=8,
                color="white" if s != "zero" else SURFACE,
            )
        left += count
ax.set_yticks(range(len(matrices)), list(matrices))
ax.invert_yaxis()
ax.axvline(0, color=RULE, linewidth=1, zorder=1)
ax.set_xlabel("λ / max|λ|")
ax.set_title("Eigenvalues, coloured by sign", fontsize=10)
axb.set_yticks(range(len(matrices)), [""] * len(matrices))
axb.invert_yaxis()
axb.set_xlabel("eigenvalue count")
axb.set_title("Inertia (n₋, n₀, n₊)", fontsize=10)
axb.grid(False)
handles = [plt.Line2D([], [], linestyle="none", markersize=7, **SIGN[s]) for s in SIGN]
ax.legend(
    handles,
    list(SIGN),
    frameon=False,
    fontsize=8,
    numpoints=1,
    ncol=3,
    loc="upper center",
    bbox_to_anchor=(0.5, -0.2),
)
plt.show()

for name, A in matrices.items():
    i = mtl5.inertia(A)
    print(f"{name:32s} n+ = {i['positive']:2d}   n- = {i['negative']:2d}   n0 = {i['zero']}")

Only the first matrix could go to Cholesky or CG. The other four need an
indefinite method. `rosser()`'s zero eigenvalue also makes it singular, and its
inertia says so (n₀ = 1). Wilkinson's near-pairs plot as single dots at this
scale: the top pair of `wilkinson(13)` differs by about 2·10⁻⁶, agreement to
about six digits.

## 2. A spectrum built to order

`randsym(n, eigenvalues, seed=...)` returns Q·Λ·Qᵀ for a random orthogonal Q.
So the eigenvalues are exactly the ones requested, and only the eigenvectors
are random. Two seeds give two matrices that look nothing alike entry by entry
but share one spectrum, and therefore one inertia.

In [ ]:
wanted = [-4.0, -2.5, -1.0, -0.5, 0.0, 0.5, 1.0, 1.5, 2.0, 3.0, 4.0, 5.0]
pair = {seed: g.randsym(12, wanted, seed=seed) for seed in (2, 3)}

BLUES = ["#cce3fd", "#9dc8f7", "#6aabf1", "#338ced", "#2370cd", "#1757ac", "#0f4a96", "#0e3d7e"]
cmap = ListedColormap(np.array([plt.matplotlib.colors.to_rgb(c) for c in BLUES]))

fig, axes = plt.subplots(1, 3, figsize=(12, 3.6), constrained_layout=True)
vmax = max(np.abs(A.to_numpy()).max() for A in pair.values())
for ax, (seed, A) in zip(axes[:2], pair.items()):
    im = ax.imshow(np.abs(A.to_numpy()), cmap=cmap, vmin=0, vmax=vmax)
    ax.set_title(f"|A|, randsym(12, wanted, seed={seed})", fontsize=10)
    ax.grid(False)
    ax.set_xticks([])
    ax.set_yticks([])
fig.colorbar(im, ax=axes[:2], shrink=0.85, label="|aᵢⱼ|")

ax = axes[2]
ax.plot(wanted, wanted, color=RULE, linewidth=1.5, linestyle="--", zorder=2)
for slot, (seed, A) in enumerate(pair.items()):
    ax.scatter(
        wanted,
        mtl5.eigvalsh(A),
        s=40 if slot == 0 else 18,
        zorder=3,
        color=SERIES[slot],
        marker="o" if slot == 0 else "s",
        label=f"seed {seed}",
    )
ax.set_xlabel("requested eigenvalue")
ax.set_ylabel("computed eigenvalue")
ax.set_title("Same spectrum, different matrices", fontsize=10)
ax.legend(frameon=False, fontsize=8, loc="upper left")
plt.show()

for seed, A in pair.items():
    err = np.max(np.abs(mtl5.eigvalsh(A) - np.sort(wanted)))
    same = np.array_equal(A.to_numpy(), g.randsym(12, wanted, seed=seed).to_numpy())
    print(
        f"seed {seed}: max |computed - requested| = {err:.1e}, "
        f"inertia {mtl5.inertia(A)}, same matrix on a second call: {same}"
    )

The eigenvalues come back to within a few units in the last place, and a seed
reproduces its matrix bit for bit. That makes inertia a *controlled* variable:
choose the counts, and only the eigenvectors are left to chance.

## 3. Slicing a spectrum

By Sylvester's law, the number of eigenvalues below a shift σ equals the number
of negative pivots in an LDLᵀ factorization of A − σI. Counting those pivots
over a range of σ draws a staircase that steps up by one at each eigenvalue.
This is **spectrum slicing**, the idea behind bisection eigensolvers and the
"how many eigenvalues lie in [a, b]?" queries of large-scale solvers. It needs
factorizations only, no eigensolver.

`mtl5.ldlt` doesn't pivot. On a tridiagonal matrix such as `wilkinson(21)`,
that is the classic Sturm-sequence count.

In [ ]:
def count_below(A, sigma):
    # The number of eigenvalues below sigma: the negative pivots of A - sigma*I.
    shifted = mtl5.matrix(A - sigma * np.eye(A.shape[0]))
    return int((mtl5.ldlt(shifted).diagonal() < 0).sum())


W = g.wilkinson(21).to_numpy()
# A tight tolerance: at the default 1e-10, eigvalsh merges the top pair (below).
lam = mtl5.eigvalsh(mtl5.matrix(W), 1e-16)
sigmas = np.linspace(lam.min() - 1, lam.max() + 1, 601)
counts = [count_below(W, s) for s in sigmas]

fig, ax = plt.subplots(figsize=(10, 3.6), constrained_layout=True)
ax.step(
    sigmas,
    counts,
    where="post",
    color=SERIES[0],
    linewidth=2,
    zorder=3,
    label="negative pivots of A − σI",
)
ax.plot(
    lam,
    np.full_like(lam, -1.2),
    linestyle="none",
    marker="|",
    markersize=14,
    color=INK_2,
    label="eigenvalues (eigvalsh)",
)
ax.set_xlabel("shift σ")
ax.set_ylabel("eigenvalues below σ")
ax.set_ylim(-2.5, 22)
ax.set_title(
    "Spectrum slicing on wilkinson(21): the count steps up at every eigenvalue", fontsize=10
)
ax.legend(frameon=False, fontsize=8, loc="upper left")
plt.show()

top_two = lam[-2:]
print("top pair:", top_two, "| difference:", f"{top_two[1] - top_two[0]:.1e}")
for label, sigma in [
    ("below the pair", top_two[0] - 1e-12),
    ("between them", top_two.mean()),
    ("above the pair", top_two[1] + 1e-12),
]:
    print(f"count {label:15s} (sigma = {sigma!r}): {count_below(W, sigma)}")

Wilkinson's pairs show up as steps of two at the scale of the plot. The top
pair differs by about 7·10⁻¹⁴, agreement to 14 digits, yet a shift placed
between the two values still counts 20: exactly one eigenvalue on each side.
The pivot count resolves a gap only a few dozen float64 spacings wide. Slicing
reports *how many* eigenvalues lie in an interval, which is exactly what a
cluster needs.

Bisection turns the count into an eigensolver. To find the k-th eigenvalue,
keep the interval [lo, hi] with count(lo) < k ≤ count(hi) and halve it. Each
step gains one bit, with no eigensolver involved.

In [ ]:
def bisect(A, k, lo, hi, steps=60):
    widths = []
    for _ in range(steps):
        mid = 0.5 * (lo + hi)
        if count_below(A, mid) >= k:
            hi = mid
        else:
            lo = mid
        widths.append(hi - lo)
    return 0.5 * (lo + hi), widths


k = 11  # the 11th smallest eigenvalue of wilkinson(21)
estimate, widths = bisect(W, k, lam.min() - 1, lam.max() + 1)

fig, ax = plt.subplots(figsize=(6.4, 3.4), constrained_layout=True)
ax.semilogy(
    np.arange(1, len(widths) + 1), widths, color=SERIES[0], linewidth=2, marker="o", markersize=3
)
ax.set_xlabel("bisection step")
ax.set_ylabel("interval width")
ax.set_title(f"Bisection on the count: eigenvalue {k} of wilkinson(21)", fontsize=10)
plt.show()

print(f"bisection: {estimate:.15f}")
print(f"eigvalsh : {lam[k - 1]:.15f}")

The width halves at every step, as bisection must, until it reaches the spacing
of float64 numbers near the eigenvalue, about 50 steps from an interval of
width 23. The answer agrees with `eigvalsh` to the last digits.

The count is also an accurate eigensolver for a cluster. Bisecting for
eigenvalues 20 and 21 splits Wilkinson's top pair. `eigvalsh` at its default
tolerance of 10⁻¹⁰ does not: it stops iterating while the two still look
equal.

In [ ]:
lo, hi = lam[-3] + 1e-6, lam[-1] + 1.0
pair_bisect = np.array([bisect(W, k, lo, hi, steps=80)[0] for k in (20, 21)])
pair_default = mtl5.eigvalsh(mtl5.matrix(W))[-2:]

print(f"{'':28s} {'eigenvalue 20':>20s} {'eigenvalue 21':>20s} {'gap':>9s}")
for label, pr in [
    ("bisection on the count", pair_bisect),
    ("eigvalsh, tol=1e-16", lam[-2:]),
    ("eigvalsh, default tol=1e-10", pair_default),
]:
    print(f"{label:28s} {pr[0]:20.15f} {pr[1]:20.15f} {pr[1] - pr[0]:9.1e}")

## 4. Inertia in low precision

A factorization computed in a format is backward stable: it is the exact
factorization of a nearby matrix A + E, with ‖E‖ of order u‖A‖. Its inertia is
the inertia of A + E. By Weyl's inequality, every eigenvalue of A + E is within
‖E‖ of one of A's. So an eigenvalue with |λ| well above u‖A‖ keeps its sign,
and one below that can come out with either sign.

The experiment builds 12×12 matrices with `randsym`, with inertia (8, 4, 0).
One eigenvalue is a small positive δ, and the others have magnitude between 0.2
and 1. For each format and each δ, LDLᵀ runs in that format on 20 seeded
matrices. The chart shows how many of the 20 report the inertia correctly. A
pivot that rounds to exactly zero, which `mtl5.ldlt` reports as an error,
counts as wrong: the format then sees a singular matrix.

In [ ]:
deltas = np.logspace(-1, -17, 17)
seeds = range(20)
others = [-1.0, -0.7, -0.4, -0.2, 0.2, 0.4, 0.6, 0.8, 0.9, 1.0, 1.0]


def ldlt_inertia(A, dtype):
    try:
        d = np.asarray(mtl5.ldlt(mtl5.convert(A, dtype)).diagonal())
    except RuntimeError:
        # A pivot rounded to exactly zero: the format sees a singular matrix,
        # n0 >= 1, which is the wrong inertia here.
        return None
    return int((d > 0).sum()), int((d < 0).sum())


correct = np.zeros((len(FORMATS), len(deltas)), dtype=int)
for j, delta in enumerate(deltas):
    for seed in seeds:
        A = g.randsym(12, others + [float(delta)], seed=100 + seed).to_numpy()
        for i, name in enumerate(FORMATS):
            correct[i, j] += ldlt_inertia(A, name) == (8, 4)

fig, ax = plt.subplots(figsize=(11, 3.9), constrained_layout=True)
im = ax.imshow(correct, cmap=cmap, vmin=0, vmax=len(seeds), aspect="auto")
ax.set_xticks(range(len(deltas)), [f"1e{int(round(np.log10(d)))}" for d in deltas], fontsize=8)
ax.set_yticks(range(len(FORMATS)), FORMATS)
ax.set_xlabel("the small eigenvalue δ (the others have |λ| ≥ 0.2, ‖A‖ = 1)")
ax.grid(False)
for r in range(correct.shape[0]):
    for c in range(correct.shape[1]):
        v = correct[r, c]
        ax.text(
            c,
            r,
            str(v),
            ha="center",
            va="center",
            fontsize=7.5,
            color="white" if v >= len(seeds) / 2 else INK,
        )
fig.colorbar(im, ax=ax, shrink=0.9, label=f"correct inertia, of {len(seeds)}")
ax.set_title(
    "LDLᵀ inertia in each format: when does a small eigenvalue's sign survive?", fontsize=11
)
plt.show()

Each row stays at or near 20 of 20 while δ is above the format's unit
roundoff. Once δ drops below it, the row falls to a plateau of 8 to 16 out of
20: the sign of δ is then decided by rounding, not by the matrix.

- **16 bits:** mostly right down to δ = 10⁻³, unreliable from 10⁻⁴. Their u is
  2.4·10⁻⁴ (posit16, takum16) and 4.9·10⁻⁴ (fp16).
- **32 bits:** f32 starts to slip at 10⁻⁶ and is at the plateau from 10⁻⁸
  (u = 6·10⁻⁸). posit32 and takum32 hold out a decade longer, to 10⁻⁸, and
  reach the plateau at 10⁻⁹ (u = 3.7·10⁻⁹).
- **f64:** correct down to δ ≈ 10⁻¹⁵.
- **`dd_cascade`:** fails only at δ = 10⁻¹⁷. Its arithmetic isn't the cause:
  `randsym` builds the matrix in float64, and rounding Q·Λ·Qᵀ to float64 moves
  every eigenvalue by about 10⁻¹⁶. Below that, the matrix being factored no
  longer has the requested eigenvalue.

The practical reading: an inertia count from a low-precision factorization
means only what the format can resolve. Eigenvalues smaller than roughly u‖A‖,
times a modest factor, have no reliable sign in that format. This matters to
any method that branches on inertia: deciding whether to use Cholesky,
checking second-order optimality conditions, or slicing a spectrum.

## 5. Gershgorin discs

Every eigenvalue lies in the union of the discs centred at aᵢᵢ with radius
Σⱼ≠ᵢ |aᵢⱼ|. The bound costs one pass over the matrix and needs no
factorization at all. How useful it is depends on how close to diagonal the
matrix is:

- **`wilkinson(11)`** is tridiagonal with a dominant diagonal, so the discs sit
  near the eigenvalues.
- **`clement(8)`** has a zero diagonal, so every disc is centred at 0.
- **`lehmer(8)`** is dense, and its discs reach far beyond its spectrum.
- **`frank(8)`** is non-symmetric (upper Hessenberg), and its discs are
  generous.

In [ ]:
gersh = {
    "wilkinson(11)": g.wilkinson(11),
    "clement(8)": g.clement(8),
    "lehmer(8)": g.lehmer(8),
    "frank(8)": g.frank(8),
}

fig, axes = plt.subplots(1, 4, figsize=(13, 3.6), constrained_layout=True)
for ax, (name, A) in zip(axes, gersh.items()):
    M = A.to_numpy()
    centres = np.diag(M)
    radii = np.abs(M).sum(axis=1) - np.abs(centres)
    for c, r in zip(centres, radii):
        ax.add_patch(
            Circle((c, 0), r, facecolor=SERIES[0], alpha=0.08, edgecolor=SERIES[0], linewidth=1)
        )
    ev = mtl5.eigvals(A)
    ax.scatter(
        ev.real,
        ev.imag,
        s=26,
        color=SERIES[1],
        marker="D",
        zorder=3,
        edgecolor=SURFACE,
        linewidth=1,
        label="eigenvalues",
    )
    lo, hi = (centres - radii).min(), (centres + radii).max()
    half = 0.5 * (hi - lo) * 1.08
    mid = 0.5 * (hi + lo)
    ax.set_xlim(mid - half, mid + half)
    ax.set_ylim(-half, half)
    ax.set_aspect("equal")
    ax.set_title(name, fontsize=10)
    ax.set_xlabel("Re λ")
    reach = max(abs(lo), abs(hi)) / np.max(np.abs(ev))
    ax.text(
        0.03,
        0.10,
        f"radius bound / spectral radius: {reach:.1f}",
        transform=ax.transAxes,
        fontsize=8,
        color=INK_2,
    )
    ax.text(
        0.03,
        0.03,
        f"lowest disc edge {lo:.1f}, smallest Re λ {ev.real.min():.2f}",
        transform=ax.transAxes,
        fontsize=8,
        color=INK_2,
    )
axes[0].set_ylabel("Im λ")
axes[0].legend(frameon=False, fontsize=8, loc="upper left")
plt.show()

The discs (blue) always contain the eigenvalues (orange), but what they tell
you depends on which end of the spectrum you ask about:

- **The largest eigenvalue** is bounded well for three of the four: the
  outermost disc edge is within 10% of the spectral radius. `frank(8)` is the
  exception, with a bound almost twice the true radius.
- **The smallest eigenvalues** are where the discs say little. `lehmer(8)` is
  SPD with λ_min ≈ 0.09, but its discs reach −3.2, so they can't certify that
  it is definite. Only a diagonally dominant matrix with a positive diagonal
  gets that certificate from Gershgorin.

So Gershgorin is the cheapest bound on the spectral radius, and a
definiteness test only for nearly diagonal matrices. To decide inertia in
general, factor the matrix (section 3) or compute the eigenvalues (section 1).

## Still to come

- **Parametric indefinite matrices.** MTL5's
  `randsym(n, kappa, mode, num_negative)` (stillwater-sc/mtl5#961) sets κ, the
  mode and the inertia together. It joins this notebook once the MTL5 release
  that carries it is bound here.
- The rest of #166: non-normality and pseudospectra, the stability of
  factorizations, and Krylov convergence against the shape of the spectrum.